Train Models

Model 1 : Logistic Regression (Baseline)
- Simple, interpretable, fast
- Good starting point to understand features effects
- class_weight="balanced" compensates for classes imbalance

In [1]:
import pandas as pd

X_train_scaled = pd.read_parquet("../data/processed/X_train.parquet", engine="pyarrow")
X_test_scaled = pd.read_parquet("../data/processed/X_test.parquet", engine="pyarrow")
y_train = pd.read_parquet("../data/processed/y_train.parquet", engine="pyarrow")["loan_status"]
y_test = pd.read_parquet("../data/processed/y_test.parquet", engine="pyarrow")["loan_status"]
print(f"X train : {X_train_scaled.shape}")
print(f"X test : {X_test_scaled.shape}")
print(f"y train : {y_train.shape}")
print(f"y test : {y_test.shape}")
print(type(y_train))

X train : (25341, 22)
X test : (6336, 22)
y train : (25341,)
y test : (6336,)
<class 'pandas.Series'>


In [2]:
print(list(X_train_scaled.columns) == list(X_test_scaled.columns))

True


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve
)
from catboost import CatBoostClassifier

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)
lr_model.fit(X_train_scaled, y_train)

lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_auc = roc_auc_score(y_test, lr_probs)

print(f'✅ Logistic Regression trained')
print(f'   ROC-AUC: {lr_auc:.4f}')

✅ Logistic Regression trained
   ROC-AUC: 0.8771


Model 2 : CatBoost

- Gradient boosting — handles non-linear relationships
- More powerful than Logistic Regression for tabular data
- scale_pos_weight adjusts for class imbalance

In [4]:
# class imbalance ratio
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos

cb_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    scale_pos_weight=scale,
    eval_metric="AUC",
    random_seed=42,
    verbose=0
)
cb_model.fit(X_train_scaled, y_train)

cb_probs = cb_model.predict_proba(X_test_scaled)[:,1]
cb_auc   = roc_auc_score(y_test, cb_probs)

print(f'✅ CatBoost trained')
print(f'   ROC-AUC: {cb_auc:.4f}')
print(f'\n💡 AUC > 0.5 = better than random. AUC = 1.0 = perfect. Target: > 0.80')

✅ CatBoost trained
   ROC-AUC: 0.9426

💡 AUC > 0.5 = better than random. AUC = 1.0 = perfect. Target: > 0.80


In [5]:
train_auc = roc_auc_score(y_train, cb_model.predict_proba(X_train_scaled)[:, 1])
print("Train AUC:", round(train_auc, 4), "| Test AUC:", round(cb_auc, 4))

Train AUC: 0.9559 | Test AUC: 0.9426


In [6]:
from pathlib import Path

Path("../models").mkdir(exist_ok=True)
Path("../data/processed").mkdir(parents=True, exist_ok=True)

In [7]:
import joblib

# Logistic Regression 
joblib.dump(lr_model, "../models/lr_model.pkl")

# CatBoost
cb_model.save_model("../models/cb_model.cbm")

In [8]:
# Save hasil prediksi data test
import pandas as pd

predicitions = pd.DataFrame({
    "y_true": y_test.values,
    "lr_prob": lr_probs,
    "cb_prob": cb_probs
}, index=y_test.index)

predicitions.to_parquet("../data/processed/predictions.parquet", engine="pyarrow", compression="snappy")

In [9]:
lr_loaded = joblib.load("../models/lr_model.pkl")

cb_loaded = CatBoostClassifier()
cb_loaded.load_model("../models/cb_model.cbm")

CatBoostClassifier(class_weights=[1, 3.641208791], depth=6, eval_metric='AUC', iterations=300, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=0)